# FRASER ライブラリ検証 demo notebook

このノートブックは `market.fraser` ライブラリ（FRASER REST API クライアント）の動作確認用 thin demo です。
FOMC Minutes / Beige Book を実 API から取得し、`data/raw/fraser/` への保存と SQLite キャッシュ統合を確認します。

## 前提条件

- `FRASER_API_KEY` が `.env` に設定済み（取得方法は `src/market/fraser/README.md` を参照）
- インターネット接続（`https://fraser.stlouisfed.org/api/` にアクセス可能）
- `uv sync --all-extras` 完了済み

## scope

本 notebook は **ライブラリ動作確認** のみを目的とした thin demo です。NLP 前処理・センチメント分析・エンベディング投入などの下流処理は `notebook/FILING_NLP/` 側に委譲します（最終セル参照）。

In [ ]:
from __future__ import annotations

from pathlib import Path

from dotenv import load_dotenv

from market.fraser import (
    BeigeBookFetcher,
    FOMCMinutesFetcher,
    FraserClient,
)

# .env から FRASER_API_KEY を環境変数として読込（FraserClient() が自動参照）
load_dotenv()

client = FraserClient()
print(f"FraserClient initialised: base_url={client.config.base_url}")

In [ ]:
# Demo 1: FOMC Minutes — 2024年の一覧取得 + 先頭1件のテキストDL
fetcher = FOMCMinutesFetcher(client=client)
meetings = fetcher.list_minutes((2024, 2024))
print(f"Found {len(meetings)} FOMC Minutes in 2024")

if meetings:
    path, meeting = fetcher.fetch_text(meetings[0].item_id, prefer="txt")
    assert isinstance(path, Path)
    print(f"Saved: {path} ({path.stat().st_size:,} bytes)")
    print(f"meeting_date: {meeting.meeting_date}")

In [ ]:
# Demo 2: Beige Book — 2024年分を並列DL（max_workers=4、30 req/min レート制限を尊重）
bb_fetcher = BeigeBookFetcher(client=client)
results = bb_fetcher.fetch_all((2024, 2024), max_workers=4)

ok = sum(1 for v in results.values() if isinstance(v, Path))
print(f"Downloaded {ok} / {len(results)} Beige Book reports")
for item_id, value in list(results.items())[:3]:
    if isinstance(value, Path):
        print(f"  - item_id={item_id}: {value} ({value.stat().st_size:,} bytes)")
    else:
        print(f"  - item_id={item_id}: ERROR {type(value).__name__}: {value}")

## 次ステップ — 下流 NLP パイプライン接続点

本 notebook は FRASER ライブラリの動作確認のみを対象とした thin demo です。
下流の NLP パイプライン（FinBERT センチメント、エンベディング投入、レジーム接続など）は別レイヤーで実装されます:

- [`notebook/FILING_NLP/`](../FILING_NLP/) — SEC Filings NLP パイプライン（FinBERT センチメント・チャンク化・埋め込み）。FRASER で取得した FOMC / Beige Book テキストも同パイプラインに投入可能です。
- `src/embedding/` — ChromaDB 投入パッケージ。FRASER 文書をベクトル DB に登録する場合の入口です。
- `src/factor/`（将来）— regime-switching 因子と FOMC センチメントを統合する factor 実装の予定。

### 動作確認チェックリスト

- [ ] `data/raw/fraser/fomc/minutes/` に txt または pdf が保存されている
- [ ] `data/raw/fraser/beige_book/` に複数の PDF / TXT が保存されている
- [ ] `data/cache/market_data.db` の `cache` テーブルに `fraser:` プレフィクスのキーが存在する

### API key 運用

API key の revoke + 再発行手順は `src/market/fraser/README.md` の Troubleshooting セクション「API キーを失効・再発行したい」を参照してください。